# Covariates for the ALT causal analysis (Google Earth Engine)

Runs `gee_causal_features.py` (same folder in the repo, `Scripts/covariates/`). It extracts the
treatments, confounders and mediators of the manuscript DAG tables for every site of the
multi-source ALT dataset, then writes the `ALDI_withYearlyData_augmented*.csv` files that
`ALDI_causality.ipynb` reads (in `Index/multisource/`).

Before running, put these in Google Drive:

| File | Drive location |
|---|---|
| `gee_causal_features.py` | `UND/Index/scripts/` |
| `alt_multisource_for_ALDI.csv` | `UND/Index/alt_multisource/` (already there for the ALDI notebook) |
| `aldi_multisource_results.csv` | `UND/Index/` (written by `ALDI_Calculation_final.ipynb`) |
| `permaice.shp` with `.shx`, `.dbf`, `.prj` | `UND/Index/ggd318_map_circumarctic/` |

Steps 1 to 3 use Earth Engine. Steps 4 and 5 only read files and can be re-run at any time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install earthengine-api geopandas scikit-learn

In [ ]:
import sys

INDEX   = '/content/drive/MyDrive/UND/Index/'
EXPORTS = '/content/drive/MyDrive/ALDI_GEE_exports'   # Earth Engine writes the CSV files here
PROJECT = 'your-earth-engine-project'                 # <- your Earth Engine Cloud project id

ALT_CSV     = INDEX + 'alt_multisource/alt_multisource_for_ALDI.csv'
ALDI_CSV    = INDEX + 'aldi_multisource_results.csv'
IPA_SHP     = INDEX + 'ggd318_map_circumarctic/permaice.shp'
FEATURES    = INDEX + 'multisource/ALDI_causal_features_complete.csv'
OUT_DIR     = INDEX + 'multisource/'

sys.path.insert(0, INDEX + 'scripts')

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project=PROJECT)

import importlib, gee_causal_features as gcf
importlib.reload(gcf)

sites = gcf.load_sites(ALT_CSV)          # sites with ALT values from 1990 (CONFIG['first_year'])
print(len(sites), 'sites')
sites.head()

## 1. Quick test

Three sites, one year, every variable, computed directly (about a minute). Check that the
values look sensible before starting the full run (for example TDD of a few hundred to
about 1500 degree-days, FDD of about 1000 to 7000, NDVI between 0 and 0.8).

In [ ]:
test = gcf.quick_test(sites, year=2015, n_sites=3)

## 2. Start the exports

One Earth Engine task per data group and year (about 150 tasks). They run on Google's side,
so Colab can be closed once they are started. Running this cell again only starts the tasks
whose file is not yet in Drive and that are not running.

In [ ]:
gcf.export_all(sites, drive_dir=EXPORTS)

## 3. Follow the tasks

Re-run this cell from time to time, or look at https://code.earthengine.google.com/tasks.
If some tasks failed, run step 2 again: it restarts only the missing ones.

In [ ]:
gcf.task_status()

## 4. Merge the exports and add the offline features

Adds distance to the coast, continentality, the AVHRR-to-MODIS adjustment of NDVI and LAI,
fire-day counts and the IPA permafrost map classes. Prints the share of filled values per
column.

In [ ]:
features = gcf.assemble_features(sites, EXPORTS, ipa_shapefile=IPA_SHP, out_csv=FEATURES)

## 5. Files for ALDI_causality.ipynb

Writes `ALDI_withYearlyData_augmented.csv` and the ALDI-class and permafrost-zone subsets to
`Index/multisource/`, and the list of all columns with their definitions
(`feature_dictionary.csv`).

In [ ]:
gcf.build_augmented(features, ALT_CSV, ALDI_CSV, OUT_DIR)
gcf.write_dictionary(OUT_DIR + 'feature_dictionary.csv')